In [7]:
import os,json
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import joblib
def expand_datetime_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    date_cols = X.select_dtypes(include=['datetime', 'datetimetz'])
    for c in date_cols:
        X[f'{c}_year'] = X[c].dt.year.astype('float64')
        X[f'{c}_month'] = X[c].dt.month.astype('float64')
        X[f'{c}_day'] = X[c].dt.day.astype('float64')
        X[f'{c}_dow'] = X[c].dt.dayofweek.astype('float64')
        X[f'{c}_is_month_end'] = X[c].dt.is_month_end.astype('float64')
    if len(date_cols) > 0:
        X = X.drop(columns=date_cols)
    return X
# Paths
DATA_PATH = r'ME_Billable_Customer_Projs-2025.xlsx'
ARTIFACT_DIR = Path('model_artifacts')
MODEL_PATH = ARTIFACT_DIR / 'gb_rag_status.pkl'
FEATURES_PATH = ARTIFACT_DIR / 'gb_expected_features.json'
# Dataset
data = pd.read_excel(DATA_PATH)
df = pd.DataFrame(data)
# Build features and targets
target_cols = ['Overall Status','Revenue Status', 'Schedule Status','Client Blockage Status']
y = df[target_cols]
X = df.drop(columns=target_cols)
# 1) expand and drop datetime columns
X = expand_datetime_features(X)
# Impute numerics then one-hot encode categoricals (train-time!)
num_cols = X.select_dtypes(include='number').columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median(numeric_only=True))
X = pd.get_dummies(X, dummy_na=True)
# Persist the exact training feature schema
expected_features = X.columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.3,stratify=y['Overall Status'])

base = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.03, max_depth=5, random_state=42)
model = MultiOutputClassifier(base)
model.fit(X_train, y_train)

# Step 4: Evaluate model
y_pred = model.predict(X_test)
for i, col in enumerate(y_test.columns):
    print(col)
    print(classification_report(y_test[col], y_pred[:, i]))

# Persist artifacts for future reuse (no retraining needed next time)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH)
with open(FEATURES_PATH, 'w', encoding='utf-8') as f:
    json.dump(expected_features, f)
print(f"\nSaved model to: {MODEL_PATH}")
print(f"Saved feature schema to: {FEATURES_PATH}")

Overall Status
              precision    recall  f1-score   support

       Green       0.94      1.00      0.97        58
         Red       1.00      0.40      0.57         5
      Yellow       1.00      0.91      0.95        11

    accuracy                           0.95        74
   macro avg       0.98      0.77      0.83        74
weighted avg       0.95      0.95      0.94        74

Revenue Status
              precision    recall  f1-score   support

       Green       0.97      1.00      0.99        70
         Red       1.00      0.50      0.67         4

    accuracy                           0.97        74
   macro avg       0.99      0.75      0.83        74
weighted avg       0.97      0.97      0.97        74

Schedule Status
              precision    recall  f1-score   support

       Green       0.98      0.98      0.98        62
         Red       1.00      1.00      1.00         2
      Yellow       0.90      0.90      0.90        10

    accuracy                